# Unit 1: Logistics - QAOA for MaxCut

**Companion notebook for *What Quantum Computers Are Actually For***

**Notebook class:** faithful worked example


**Code note:** This notebook is written as teaching code rather than library code. The Quokka calls, circuit construction, and measurement post-processing stay explicit on purpose so you can inspect the mechanism end to end. A production library would factor more of this behind helpers; here we keep the moving parts visible.

This notebook uses the smallest nontrivial MaxCut instance - a triangle graph - and a depth-1 QAOA circuit that is exact for this case on Quokka.

**What you'll see:**
1. Define a graph and its MaxCut cost function
2. Solve it classically (brute-force) to know the right answer
3. Build the QAOA circuit as a QASM string
4. Run it on Quokka and verify that optimal cut states dominate
5. Sweep QAOA parameters to visualise the optimisation landscape

In [ ]:
import json
import itertools
from collections import Counter

import numpy as np
import matplotlib.pyplot as plt
import requests
from requests.packages.urllib3.exceptions import InsecureRequestWarning
requests.packages.urllib3.disable_warnings(InsecureRequestWarning)

# --- Quokka connection ---
QUOKKA = "quokka1"  # Change to quokka2-6 if this one is offline
QUOKKA_URL = f"http://{QUOKKA}.quokkacomputing.com/qsim/qasm"

def _decode_quokka_counts(payload: dict) -> dict:
    """Normalize Quokka responses to a simple {bitstring: count} mapping."""
    if isinstance(payload, dict) and all(isinstance(v, int) for v in payload.values()):
        return payload

    if not isinstance(payload, dict):
        raise TypeError(f"Unexpected Quokka payload type: {type(payload)!r}")

    if payload.get("error_code", 0) not in (0, None):
        raise RuntimeError(f"Quokka error {payload.get('error_code')}: {payload.get('error')}")

    result = payload.get("result")
    if not isinstance(result, dict) or "c" not in result:
        raise ValueError(f"Unexpected Quokka result format: {payload}")

    counts = {}
    for shot in result["c"]:
        bitstring = ''.join(str(int(bit)) for bit in shot)
        counts[bitstring] = counts.get(bitstring, 0) + 1
    return counts

def run_qasm(program: str, shots: int = 1024) -> dict:
    """Send a QASM program to a cloud Quokka and return counts by bitstring."""
    result = requests.post(QUOKKA_URL, json={"script": program, "count": shots}, verify=False)
    result.raise_for_status()
    return _decode_quokka_counts(result.json())

print(f"Connected to {QUOKKA_URL}")

## 1. Define the graph

A triangle: 3 nodes, 3 edges. The optimal MaxCut is 2 (colour one node differently from the other two).

```
    0
   / \
  1 — 2
```

In [ ]:
# Graph definition
n_qubits = 3
edges = [(0, 1), (0, 2), (1, 2)]

def cut_value(bitstring: str, edges: list) -> int:
    """Count how many edges are cut by this colouring."""
    return sum(1 for i, j in edges if bitstring[i] != bitstring[j])

# Brute-force: evaluate all 2^n colourings
print("Bitstring | Cut value")
print("-" * 22)
best_cut = 0
for bits in itertools.product('01', repeat=n_qubits):
    bs = ''.join(bits)
    cv = cut_value(bs, edges)
    best_cut = max(best_cut, cv)
    print(f"   {bs}    |    {cv}")

print(f"\nOptimal cut value: {best_cut}")

## 2. Build the QAOA circuit

The QAOA circuit for MaxCut at depth $p=1$:

1. **Hadamard** all qubits → equal superposition
2. **Problem unitary** $e^{-i\gamma C}$: for each edge $(i,j)$, apply CNOT–$R_Z(2\gamma)$–CNOT
3. **Mixer** $e^{-i\beta B}$: apply $R_X(2\beta)$ to each qubit
4. **Measure**

In [ ]:
def qaoa_qasm(n: int, edges: list, gamma: float, beta: float) -> str:
    """Build a depth-1 QAOA MaxCut circuit in OpenQASM 2.0."""
    lines = [
        'OPENQASM 2.0;',
        'include "qelib1.inc";',
        f'qreg q[{n}];',
        f'creg c[{n}];',
        '',
        '// Step 1: Superposition',
    ]
    for i in range(n):
        lines.append(f'h q[{i}];')

    lines.append('')
    lines.append('// Step 2: Problem unitary exp(-i*gamma*C)')
    for i, j in edges:
        lines.append(f'cx q[{i}], q[{j}];')
        lines.append(f'rz({2*gamma:.6f}) q[{j}];')
        lines.append(f'cx q[{i}], q[{j}];')

    lines.append('')
    lines.append('// Step 3: Mixer exp(-i*beta*B)')
    for i in range(n):
        lines.append(f'rx({2*beta:.6f}) q[{i}];')

    lines.append('')
    lines.append('// Step 4: Measure')
    for i in range(n):
        lines.append(f'measure q[{i}] -> c[{i}];')

    return chr(10).join(lines)

# Exact depth-1 optimum for the triangle MaxCut instance.
gamma_opt = 1.264491043069892
beta_opt = 0.3063052837250049
circuit = qaoa_qasm(n_qubits, edges, gamma_opt, beta_opt)
print(circuit)

## 3. Run on Quokka

In [ ]:
# Run the QAOA circuit
shots = 1024
results = run_qasm(circuit, shots=shots)

# Annotate with cut values
print(f"{'Outcome':>8} | {'Counts':>6} | {'Cut value':>9}")
print("-" * 32)
for bitstring in sorted(results.keys()):
    count = results[bitstring]
    cv = cut_value(bitstring, edges)
    marker = " ← optimal" if cv == best_cut else ""
    print(f"{bitstring:>8} | {count:>6} | {cv:>9}{marker}")

In [ ]:
# Visualise: colour bars by cut value
labels = sorted(results.keys())
counts = [results[k] for k in labels]
colors = ['#009688' if cut_value(k, edges) == best_cut else '#B0BEC5' for k in labels]

plt.figure(figsize=(10, 5))
plt.bar(labels, counts, color=colors)
plt.xlabel('Measurement outcome')
plt.ylabel('Counts')
plt.title(f'QAOA MaxCut on a triangle (γ={gamma_opt:.3f}, β={beta_opt:.3f}, {shots} shots)')
plt.legend(handles=[
    plt.Rectangle((0,0),1,1, color='#009688', label=f'Optimal (cut={best_cut})'),
    plt.Rectangle((0,0),1,1, color='#B0BEC5', label='Suboptimal'),
])
plt.tight_layout()
plt.show()

# Compute expected cut value
total_shots = sum(counts)
expected_cut = sum(cut_value(k, edges) * results[k] / total_shots for k in results)
print(f"Expected cut value: {expected_cut:.3f} (optimal: {best_cut})")
print(f"Approximation ratio: {expected_cut / best_cut:.3f}")

## 4. Parameter landscape

Sweep $\gamma$ and $\beta$ to visualise how the expected cut value depends on the QAOA parameters. This is the landscape the classical optimiser navigates.

In [ ]:
# Sweep gamma and beta
n_points = 15
gammas = np.linspace(0, np.pi, n_points)
betas = np.linspace(0, np.pi / 2, n_points)
landscape = np.zeros((n_points, n_points))

print(f"Running {n_points * n_points} circuits on Quokka...")
for gi, g in enumerate(gammas):
    for bi, b in enumerate(betas):
        qasm = qaoa_qasm(n_qubits, edges, g, b)
        res = run_qasm(qasm, shots=256)
        total = sum(res.values())
        landscape[bi, gi] = sum(cut_value(k, edges) * res[k] / total for k in res)
    print(f"  γ = {g:.2f} done")

print("Done!")

In [ ]:
# Plot the landscape
plt.figure(figsize=(8, 6))
plt.imshow(landscape, origin='lower', aspect='auto',
           extent=[0, np.pi, 0, np.pi/2],
           cmap='viridis', vmin=0, vmax=best_cut)
plt.colorbar(label='Expected cut value')
plt.xlabel('γ')
plt.ylabel('β')
plt.title('QAOA Parameter Landscape (depth p=1)')
plt.plot(gamma_opt, beta_opt, 'r*', markersize=15, label=f'Optimum (γ={gamma_opt:.2f}, β={beta_opt:.2f})')
plt.legend()
plt.tight_layout()
plt.show()

## 5. Classical comparison

How does QAOA compare to just sampling random colourings?

In [ ]:
# Random sampling baseline
n_random = 1024
random_cuts = []
for _ in range(n_random):
    bs = ''.join(np.random.choice(['0', '1']) for _ in range(n_qubits))
    random_cuts.append(cut_value(bs, edges))

print(f"Random sampling: expected cut = {np.mean(random_cuts):.3f}")
print(f"QAOA:            expected cut = {expected_cut:.3f}")
print(f"Optimal:         expected cut = {best_cut:.3f}")
print()
print(f"QAOA improvement over random: {(expected_cut - np.mean(random_cuts)) / np.mean(random_cuts) * 100:.1f}%")

## What's next?

- **Bigger graph:** Try a 4-node or 5-node graph by editing the `edges` list and `n_qubits`.
- **Deeper QAOA:** Extend `qaoa_qasm` to depth $p=2$ by adding a second round of problem + mixer unitaries with separate parameters.
- **Real optimisation:** Replace the fixed $\gamma, \beta$ with `scipy.optimize.minimize` to find the best parameters automatically.
- **The QASM on Quokka:** See the corresponding deep dive chapter for the pure QASM version.
- **The next unit:** [Unit 2 - Cryptography](https://github.com/johnazariah/quantum-bottleneck/blob/main/manuscript/03-cryptography.md) - where Shor's algorithm breaks the trapdoor.